# LeetCode 3352 — Count K-Reducible Numbers Less Than N bằng Python

- Link: https://leetcode.com/problems/count-k-reducible-numbers-less-than-n/
- Độ khó: **Hard**
- Chủ đề: **Đếm theo chữ số nhị phân (digit DP / tổ hợp)**, nhận xét **sau một thao tác số chỉ còn ≤ 800** ⇒ tiền xử lý số bước cho mọi popcount

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countKReducibleNumbers(self, s: str, k: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra: sau **thao tác đầu tiên**, `x` biến thành `popcount(x) ≤ len(s) ≤ 800` — nhỏ! ⇒ tiền xử lý `steps[c]` = số thao tác để `c` về 1.
2. Điều kiện `x` là k-reducible ⇔ `x = 1` hoặc `1 + steps[popcount(x)] ≤ k` ⇔ `steps[popcount(x)] ≤ k − 1` (với `x = 1` cũng đúng vì `steps[1] = 0`).
3. Quy về: **đếm số `x ∈ [1, n)` có đúng `c` bit 1**, cho mọi `c` — kỹ thuật "đi theo tiền tố của `n`" với tổ hợp `C(rem, j)`.
4. Viết 3 cách: digit DP đệ quy, tổ hợp Pascal, DP lặp theo tiền tố.
5. Kiểm chứng bằng vét cạn với `n` nhỏ.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` biểu diễn `n`, và số nguyên `k`. Số nguyên `x` gọi là **k-reducible** nếu thực hiện **tối đa `k` lần** thao tác "thay `x` bằng số bit 1 của nó" thì thu được `1`. Đếm số nguyên dương **nhỏ hơn `n`** là k-reducible, modulo `10^9 + 7`.

**Ví dụ 1:** `s = "111"`, `k = 1` → `3` (`1, 2, 4`).
**Ví dụ 2:** `s = "1000"`, `k = 2` → `6` (`1..6`).
**Ví dụ 3:** `s = "1"`, `k = 3` → `0`.

**Ràng buộc:** `1 ≤ len(s) ≤ 800`, không có số 0 đầu; `1 ≤ k ≤ 5`.

In [1]:
from typing import List
from functools import lru_cache
MOD = 10 ** 9 + 7

examples = [("111", 1, 3), ("1000", 2, 6), ("1", 3, 0)]
for s, k, exp in examples:
    print(f"s = {s!r} (n = {int(s, 2)}), k = {k} -> kỳ vọng {exp}")

s = '111' (n = 7), k = 1 -> kỳ vọng 3
s = '1000' (n = 8), k = 2 -> kỳ vọng 6
s = '1' (n = 1), k = 3 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Số bước rút gọn của các số nhỏ

`steps[1] = 0`, `steps[c] = 1 + steps[popcount(c)]` với `c ≥ 2`. Vì `popcount(c) < c` với `c ≥ 2`, ta tính tăng dần.

In [2]:
def steps_table(L):
    st = [0] * (L + 1)
    for c in range(2, L + 1):
        st[c] = 1 + st[bin(c).count('1')]
    return st

st = steps_table(800)
print('steps[1..16] =', st[1:17])
print('steps lớn nhất với c <= 800:', max(st[1:]), '-> ví dụ c =', st.index(max(st[1:])))

steps[1..16] = [0, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 2, 1]
steps lớn nhất với c <= 800: 4 -> ví dụ c = 127


Số bước tăng **cực chậm** (mọi `c ≤ 800` cần nhiều nhất 4 bước), nên với `x < 2^800`, `x` cần nhiều nhất 5 thao tác — khớp với `k ≤ 5`.

### 3.2. Đếm số nhỏ hơn `n` có đúng `c` bit 1

Duyệt `s` từ trái sang. Tại vị trí `i` mà `s[i] = '1'`, nếu ta đặt bit `0` ở đó (giữ nguyên tiền tố trước), số thu được chắc chắn `< n`, và `rem = L − i − 1` bit sau tự do. Nếu tiền tố đã có `ones` bit 1 thì có `C(rem, j)` số với tổng `ones + j` bit 1. Mỗi số `< n` được đếm **đúng một lần** — tại vị trí bit đầu tiên nó khác `n`.

In [3]:
from math import comb

def popcount_distribution(s):
    L = len(s)
    cnt = [0] * (L + 1)
    ones = 0
    for i, ch in enumerate(s):
        if ch == '1':
            rem = L - i - 1
            for j in range(rem + 1):
                cnt[ones + j] += comb(rem, j)
            ones += 1
    return cnt

s = "1011"
dist = popcount_distribution(s)
real = [0] * (len(s) + 1)
for x in range(int(s, 2)):
    real[bin(x).count('1')] += 1
print('phân phối popcount của 0..n-1:', dist, '| đếm trực tiếp:', real)

phân phối popcount của 0..n-1: [1, 4, 5, 1, 0] | đếm trực tiếp: [1, 4, 5, 1, 0]


Lưu ý phân phối trên **bao gồm cả số 0** (popcount 0) — ta luôn bỏ qua `c = 0` khi cộng đáp án.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `n = 7`, `k = 1`. Cần `steps[popcount(x)] ≤ 0` ⇔ `popcount(x) = 1` ⇔ `x` là lũy thừa của 2: `1, 2, 4` ⇒ 3 ✓.

Ví dụ 2: `n = 8`, `k = 2`. Cần `steps[popcount(x)] ≤ 1` ⇔ `popcount ∈ {1, 2}` (vì `steps[2] = 1`, `steps[3] = 2`). Trong `1..7`, chỉ `7` có 3 bit ⇒ `6` ✓.

Bài toán gồm hai phần **tách rời**: (1) tập popcount "tốt" `{c : steps[c] ≤ k − 1}`; (2) đếm số `< n` theo popcount.

## 5. Cách 1 — Brute Force: mô phỏng mọi `x < n`

In [4]:
def brute(s, k):
    n = int(s, 2)
    total = 0
    for x in range(1, n):
        t = 0
        while x != 1:
            x = bin(x).count('1')
            t += 1
        total += t <= k
    return total % MOD

for s, k, exp in examples:
    print(brute(s, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
6 | kỳ vọng 6
0 | kỳ vọng 0


**Độ phức tạp:** `O(n · log n)` với `n` tới `2^800` — chỉ dùng khi `len(s) ≤ 16`.

## 6. Ý tưởng thuật toán / bước đột phá

1. `L = len(s)`; tính `steps[0..L]`.
2. Tính `cnt[c]` = số `x ∈ [0, n)` có `c` bit 1 (mục 3.2) — dùng tam giác Pascal modulo.
3. Đáp án `= Σ_{c ≥ 1, steps[c] ≤ k − 1} cnt[c] mod (10^9 + 7)`.

Độ phức tạp `O(L²)` = `640 000` phép tính với `L = 800`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Digit DP đệ quy `(i, ones, tight)`

`dfs(i, ones, tight)` = số cách điền các bit từ `i` trở đi sao cho số cuối cùng `< n` và popcount thuộc tập tốt. Khi hết bit, số chỉ hợp lệ nếu **không còn tight** (đã nhỏ hơn hẳn `n`).

In [5]:
import sys
sys.setrecursionlimit(10000)

def digit_dp(s, k):
    L = len(s)
    st = steps_table(L)
    good = [c >= 1 and st[c] <= k - 1 for c in range(L + 1)]

    @lru_cache(maxsize=None)
    def dfs(i, ones, tight):
        if i == L:
            return int(not tight and good[ones])
        limit = int(s[i]) if tight else 1
        res = 0
        for b in range(limit + 1):
            res += dfs(i + 1, ones + b, tight and b == limit)
        return res % MOD

    ans = dfs(0, 0, True)
    dfs.cache_clear()
    return ans

for s, k, exp in examples:
    print(digit_dp(s, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
6 | kỳ vọng 6
0 | kỳ vọng 0


Số trạng thái `O(L²)`; với `L = 800` đệ quy sâu 800 tầng — cần nâng giới hạn đệ quy.

### 7.2. Cách 3 — Tổ hợp với tam giác Pascal (lời giải chính)

In [6]:
def combinatorial(s, k):
    L = len(s)
    st = steps_table(L)
    C = [[0] * (L + 1) for _ in range(L + 1)]
    for i in range(L + 1):
        C[i][0] = 1
        for j in range(1, i + 1):
            C[i][j] = (C[i - 1][j - 1] + C[i - 1][j]) % MOD
    cnt = [0] * (L + 1)
    ones = 0
    for i, ch in enumerate(s):
        if ch == '1':
            rem = L - i - 1
            for j in range(rem + 1):
                cnt[ones + j] = (cnt[ones + j] + C[rem][j]) % MOD
            ones += 1
    return sum(cnt[c] for c in range(1, L + 1) if st[c] <= k - 1) % MOD

for s, k, exp in examples:
    print(combinatorial(s, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
6 | kỳ vọng 6
0 | kỳ vọng 0


### 7.3. Cách 4 — DP lặp theo tiền tố (không cần tổ hợp)

Giữ `less[c]` = số tiền tố (độ dài `i`) **đã nhỏ hơn hẳn** tiền tố của `n` và có `c` bit 1; cùng với một tiền tố "đang bằng" duy nhất có `ones` bit 1. Mỗi bit mới: tiền tố đã nhỏ hơn được nối `0` hoặc `1` tự do; tiền tố bằng nếu gặp `s[i] = '1'` thì nhánh nối `0` rơi vào `less`.

In [7]:
def prefix_dp(s, k):
    L = len(s)
    st = steps_table(L)
    less = [0] * (L + 1)
    ones = 0
    for ch in s:
        new = [0] * (L + 1)
        for c in range(L):
            if less[c]:
                new[c] = (new[c] + less[c]) % MOD          # nối bit 0
                new[c + 1] = (new[c + 1] + less[c]) % MOD  # nối bit 1
        if ch == '1':
            new[ones] = (new[ones] + 1) % MOD              # tiền tố bằng, đặt 0 thay vì 1
            ones += 1
        less = new
    return sum(less[c] for c in range(1, L + 1) if st[c] <= k - 1) % MOD

for s, k, exp in examples:
    print(prefix_dp(s, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
6 | kỳ vọng 6
0 | kỳ vọng 0


## 8. TRACE từng bước

Trace cách 3 trên `s = "10110"` (`n = 22`), `k = 2`.

In [8]:
def trace(s, k):
    L = len(s)
    st = steps_table(L)
    print('steps:', {c: st[c] for c in range(1, L + 1)}, '| tốt (steps <= k-1):',
          [c for c in range(1, L + 1) if st[c] <= k - 1])
    cnt = [0] * (L + 1)
    ones = 0
    for i, ch in enumerate(s):
        if ch == '1':
            rem = L - i - 1
            add = [comb(rem, j) for j in range(rem + 1)]
            print(f"  i = {i}: đặt 0 thay vì 1, tiền tố có {ones} bit 1, {rem} bit tự do -> thêm {add} vào popcount {ones}..{ones + rem}")
            for j in range(rem + 1):
                cnt[ones + j] += add[j]
            ones += 1
    print('  cnt (popcount 0..L):', cnt)
    ans = sum(cnt[c] for c in range(1, L + 1) if st[c] <= k - 1)
    print('đáp án:', ans, '| vét cạn:', brute(s, k))

trace("10110", 2)

steps: {1: 0, 2: 1, 3: 2, 4: 1, 5: 2} | tốt (steps <= k-1): [1, 2, 4]
  i = 0: đặt 0 thay vì 1, tiền tố có 0 bit 1, 4 bit tự do -> thêm [1, 4, 6, 4, 1] vào popcount 0..4
  i = 2: đặt 0 thay vì 1, tiền tố có 1 bit 1, 2 bit tự do -> thêm [1, 2, 1] vào popcount 1..3
  i = 3: đặt 0 thay vì 1, tiền tố có 2 bit 1, 1 bit tự do -> thêm [1, 1] vào popcount 2..3
  cnt (popcount 0..L): [1, 5, 9, 6, 1, 0]
đáp án: 15 | vét cạn: 15


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def countKReducibleNumbers(self, s: str, k: int) -> int:
        MOD = 10 ** 9 + 7
        L = len(s)
        steps = [0] * (L + 1)
        for c in range(2, L + 1):
            steps[c] = 1 + steps[bin(c).count('1')]
        C = [[0] * (L + 1) for _ in range(L + 1)]
        for i in range(L + 1):
            row, prev = C[i], C[i - 1]
            row[0] = 1
            for j in range(1, i + 1):
                row[j] = (prev[j - 1] + prev[j]) % MOD
        cnt = [0] * (L + 1)
        ones = 0
        for i, ch in enumerate(s):
            if ch == '1':
                rem = L - i - 1
                Crem = C[rem]
                for j in range(rem + 1):
                    cnt[ones + j] += Crem[j]
                ones += 1
        ans = 0
        for c in range(1, L + 1):
            if steps[c] <= k - 1:
                ans += cnt[c]
        return ans % MOD

for s, k, exp in examples:
    got = Solution().countKReducibleNumbers(s, k)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
6 OK
0 OK


## 10. Vì sao thuật toán đúng

1. **Điều kiện k-reducible:** với `x ≥ 2`, thao tác đầu tiên đưa `x` về `popcount(x) = c`; sau đó cần đúng `steps[c]` thao tác nữa (dãy là tất định). Vậy `x` k-reducible ⇔ `1 + steps[c] ≤ k`. Với `x = 1`: 0 thao tác, và `c = 1`, `steps[1] = 0 ≤ k − 1` ⇒ công thức vẫn đúng.
2. **Phân hoạch các số `< n`:** mỗi `x < n` (cùng độ dài `L`, cho phép số 0 đầu) có vị trí đầu tiên `i` khác `n`, tại đó `s[i] = 1`, `x_i = 0`. Tiền tố trước `i` trùng `n` (có `ones` bit 1), phần sau tự do ⇒ mỗi `x` được đếm đúng một lần trong `C(rem, j)`.
3. **Số 0** có popcount 0 nên bị loại khi chỉ cộng `c ≥ 1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n log n)` | `O(1)` |
| 2. Digit DP đệ quy | `O(L²)` | `O(L²)` |
| 3. Tổ hợp Pascal | `O(L²)` | `O(L²)` |
| 4. DP lặp theo tiền tố | `O(L²)` | `O(L)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'digit_dp': digit_dp, 'combinatorial': combinatorial, 'prefix_dp': prefix_dp,
         'submit': lambda s, k: Solution().countKReducibleNumbers(s, k)}
edge = [("10", 1, 1), ("11", 1, 2), ("100", 1, 2), ("1000", 1, 3), ("1111", 2, 10), ("1111", 1, 4)]
for s, k, exp in examples + edge:
    assert brute(s, k) == exp, (s, k, brute(s, k))
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3352)
cnt = 0
for _ in range(600):
    L = rng.randint(1, 12)
    s = '1' + ''.join(rng.choice('01') for _ in range(L - 1))
    k = rng.randint(1, 5)
    exp = brute(s, k)
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ (s, k).')

Khớp vét cạn trên 600 bộ (s, k).


### 12.3. Ca lớn `len(s) = 800`

In [12]:
import time
big = {
    'toàn 1, k = 1': ("1" * 800, 1),
    'ngẫu nhiên, k = 2': ('1' + ''.join(rng.choice('01') for _ in range(799)), 2),
    '1 rồi 799 số 0, k = 5': ("1" + "0" * 799, 5),
}
for name, (s, k) in big.items():
    t = time.time(); a = Solution().countKReducibleNumbers(s, k); t1 = time.time() - t
    t = time.time(); b = prefix_dp(s, k); t2 = time.time() - t
    t = time.time(); c = digit_dp(s, k); t3 = time.time() - t
    assert a == b == c
    print(f"{name:24s}: {a:10d} | nộp {t1:.2f}s | prefix_dp {t2:.2f}s | digit_dp {t3:.2f}s")

toàn 1, k = 1           :        800 | nộp 0.09s | prefix_dp 0.07s | digit_dp 0.39s


ngẫu nhiên, k = 2       :  843763931 | nộp 0.05s | prefix_dp 0.06s | digit_dp 0.28s


1 rồi 799 số 0, k = 5   :  450332441 | nộp 0.03s | prefix_dp 0.06s | digit_dp 0.29s


Kiểm tra độc lập: với `s = "1" + "0"*799` và `k = 5`, **mọi** số `1 ≤ x < 2^799` đều 5-reducible (vì `steps[c] ≤ 4` với mọi `c ≤ 800`), nên đáp án là `(2^799 − 1) mod p`; với `k = 1` đáp án là số lũy thừa của 2 nhỏ hơn `n`, tức `L − 1`.

In [13]:
s = "1" + "0" * 799
assert Solution().countKReducibleNumbers(s, 5) == (pow(2, 799, MOD) - 1) % MOD
assert Solution().countKReducibleNumbers("1" * 800, 1) == 800
assert Solution().countKReducibleNumbers(s, 1) == 799
print('2^799 - 1 và số lũy thừa của 2: đúng')

2^799 - 1 và số lũy thừa của 2: đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `steps[c] ≤ k` (quên tính thao tác đầu tiên)

In [14]:
def wrong_off_by_one(s, k):
    L = len(s)
    st = steps_table(L)
    cnt = popcount_distribution(s)
    return sum(cnt[c] for c in range(1, L + 1) if st[c] <= k) % MOD     # SAI

print('sai :', wrong_off_by_one("111", 1), '| đúng:', Solution().countKReducibleNumbers("111", 1))

sai : 6 | đúng: 3


**Vì sao sai:** `3 = "11"` cần 1 thao tác để thành `2`, rồi thêm 1 nữa để thành `1` — tổng 2 > 1. Công thức sai đếm cả `3, 5, 6`.

### Lỗi 2 — Đếm `x ≤ n` thay vì `x < n`

In [15]:
def wrong_inclusive(s, k):
    n_pc = s.count('1')
    st = steps_table(len(s))
    extra = int(st[n_pc] <= k - 1)
    return (Solution().countKReducibleNumbers(s, k) + extra) % MOD    # SAI: cộng thêm chính n

print('sai :', wrong_inclusive("100", 1), '| đúng:', Solution().countKReducibleNumbers("100", 1))

sai : 3 | đúng: 2


**Vì sao sai:** đề yêu cầu **nhỏ hơn** `n`; `n = 4` là lũy thừa 2 nhưng không được đếm. Thuật toán chuẩn tự động loại `n` vì chỉ đếm số có vị trí khác biệt đầu tiên mang bit `0`.

### Lỗi 3 — Mô phỏng `while x > 1` khiến số 0 được coi là "đã rút gọn"

In [16]:
def wrong_count_zero(s, k):
    L = len(s)
    st = steps_table(L)
    cnt = popcount_distribution(s)
    # SAI: coi popcount 0 (số 0) cũng hợp lệ vì vòng while x > 1 dừng ngay
    return sum(cnt[c] for c in range(0, L + 1) if c == 0 or st[c] <= k - 1) % MOD

print('sai :', wrong_count_zero("111", 1), '| đúng:', Solution().countKReducibleNumbers("111", 1))

sai : 4 | đúng: 3


**Vì sao sai:** đề chỉ đếm số **nguyên dương**; `0` không bao giờ về được `1`.

### Lỗi 4 — Lệch chỉ số số bit tự do: `rem = L − i` thay vì `L − i − 1`

In [17]:
def wrong_rem(s, k):
    L = len(s)
    st = steps_table(L + 1)
    cnt = [0] * (L + 2)
    ones = 0
    for i, ch in enumerate(s):
        if ch == '1':
            rem = L - i                                   # SAI: tính cả bit i
            for j in range(rem + 1):
                cnt[ones + j] += comb(rem, j)
            ones += 1
    return sum(cnt[c] for c in range(1, L + 1) if st[c] <= k - 1) % MOD

print('sai :', wrong_rem("10", 1), '| đúng:', Solution().countKReducibleNumbers("10", 1))

sai : 2 | đúng: 1


**Vì sao sai:** bit `i` đã bị **cố định là 0**, chỉ `L − i − 1` bit phía sau là tự do. Với `n = 2`, chỉ có `x = 1` thỏa — đáp án 1, không phải 2.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thao tác "thay bằng popcount" làm số **co lại cực nhanh** ⇒ chỉ thao tác đầu tiên chạm số lớn; phần còn lại tiền xử lý.
- "Đếm số `< n` (cho dạng nhị phân/thập phân) có thuộc tính phụ thuộc tổng chữ số" ⇒ digit DP hoặc tổ hợp theo vị trí khác biệt đầu tiên.
- Nhớ modulo trong tam giác Pascal để tránh số lớn (Python chịu được nhưng chậm hơn).

## 15. Biến thể của bài

1. Đếm số `≤ n` ⇒ cộng thêm 1 nếu chính `n` thỏa.
2. Đếm trong đoạn `[a, b)` ⇒ `f(b) − f(a)`.
3. Thao tác khác: "thay bằng tổng chữ số thập phân" ⇒ digit DP hệ 10, tổng chữ số ≤ `9L`.
4. Đếm số cần **đúng** `k` thao tác.

In [18]:
# Biến thể 3 và 4
def exactly_k(s, k):
    L = len(s)
    st = steps_table(L)
    cnt = popcount_distribution(s)
    total = sum(cnt[c] for c in range(1, L + 1) if 1 + st[c] == k)
    if k == 0:
        total = int(int(s, 2) > 1)        # chỉ x = 1
    elif k == 1:
        total -= int(int(s, 2) > 1)       # x = 1 cần 0 bước, không phải 1
    return total % MOD

def exactly_brute(s, k):
    c = 0
    for x in range(1, int(s, 2)):
        t = 0
        while x != 1:
            x = bin(x).count('1'); t += 1
        c += t == k
    return c

def digit_sum_steps_brute(n, k):
    c = 0
    for x in range(1, n):
        t = 0
        while x >= 10:
            x = sum(map(int, str(x))); t += 1
        c += t <= k
    return c

rng = random.Random(15)
for _ in range(300):
    L = rng.randint(1, 11)
    s = '1' + ''.join(rng.choice('01') for _ in range(L - 1))
    k = rng.randint(0, 4)
    assert exactly_k(s, k) == exactly_brute(s, k), (s, k)
print('Biến thể "đúng k thao tác" khớp vét cạn; ví dụ tổng chữ số (rút về 1 chữ số trong <= 1 bước), n = 1000:',
      digit_sum_steps_brute(1000, 1))

Biến thể "đúng k thao tác" khớp vét cạn; ví dụ tổng chữ số (rút về 1 chữ số trong <= 1 bước), n = 1000: 219


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mô phỏng mọi `x` | `O(n log n)` | Oracle |
| 2 | Digit DP đệ quy | `O(L²)` | Tổng quát, đệ quy sâu |
| 3 | Tổ hợp Pascal | `O(L²)` | **Nộp bài** |
| 4 | DP lặp tiền tố | `O(L²)`, bộ nhớ `O(L)` | Không cần `C` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "1000000"`, `k = 1`.

**Bài 2 (tính tay):** `s = "1010"`, `k = 2`.

**Bài 3 (điền TODO):** đếm số k-reducible trong đoạn `[a, b]` (cho dạng nhị phân).

**Bài 4 (tự viết lại):** liệt kê **số k-reducible lớn nhất** nhỏ hơn `n` (trả về chuỗi nhị phân).

**Bài 5 (mini challenge):** với mỗi `k = 1..5`, đếm cùng lúc (một lần tính `cnt`).

In [19]:
def range_count_todo(a, b, k):
    # TODO
    raise NotImplementedError

def largest_below_todo(s, k):
    # TODO
    raise NotImplementedError

def all_k_todo(s):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ lũy thừa của 2. (2) `1, 2, …, 32`. (3) `6`.

**Bài 2.** (1) `n = 10`, popcount tốt `{1, 2}`. (2) Loại các số có 3 bit trong `1..9`: `7`. (3) `8`.

**Bài 3.** (1) `f(b + 1) − f(a)`. (2) Cộng 1 vào chuỗi nhị phân. (3) Cẩn thận modulo âm.

**Bài 4.** (1) Thử từ `n − 1` đi xuống? Chậm. (2) Tham lam: tìm vị trí `i` muộn nhất có thể đặt `0` thay `1` rồi điền phần sau sao cho popcount tốt và lớn nhất. (3) Phần sau: đặt các bit 1 ở **đầu** phần tự do.

**Bài 5.** (1) `cnt` không phụ thuộc `k`. (2) Cộng dồn theo `steps[c]`. (3) Một vòng lặp.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().countKReducibleNumbers("1000000", 1) == 6
assert Solution().countKReducibleNumbers("1010", 2) == 8 == brute("1010", 2)

def range_count(a, b, k):
    f = lambda m: Solution().countKReducibleNumbers(bin(m)[2:], k) if m >= 1 else 0
    return (f(b + 1) - f(a)) % MOD

def range_brute(a, b, k):
    c = 0
    for x in range(max(a, 1), b + 1):
        t = 0
        while x != 1:
            x = bin(x).count('1'); t += 1
        c += t <= k
    return c

rng = random.Random(3)
for _ in range(300):
    a = rng.randint(0, 300); b = rng.randint(a, 400); k = rng.randint(1, 4)
    assert range_count(a, b, k) == range_brute(a, b, k)
print('Bài 1: 6 | Bài 2: 8 | Bài 3 khớp vét cạn.')

Bài 1: 6 | Bài 2: 8 | Bài 3 khớp vét cạn.


In [21]:
# Bài 4, 5
def largest_below(s, k):
    L = len(s)
    st = steps_table(L)
    good = [c >= 1 and st[c] <= k - 1 for c in range(L + 1)]
    ones_before = [0] * (L + 1)
    for i, ch in enumerate(s):
        ones_before[i + 1] = ones_before[i] + (ch == '1')
    for i in range(L - 1, -1, -1):          # vị trí khác biệt muộn nhất cho số lớn nhất
        if s[i] != '1':
            continue
        rem, ones = L - i - 1, ones_before[i]
        for j in range(rem, -1, -1):        # nhiều bit 1 hơn ở đầu phần tự do => lớn hơn
            if good[ones + j]:
                return (s[:i] + '0' + '1' * j + '0' * (rem - j)).lstrip('0') or None
    return None

def largest_brute(s, k):
    for x in range(int(s, 2) - 1, 0, -1):
        y, t = x, 0
        while y != 1:
            y = bin(y).count('1'); t += 1
        if t <= k:
            return bin(x)[2:]
    return None

def all_k(s):
    L = len(s)
    st = steps_table(L)
    cnt = popcount_distribution(s)
    return [sum(cnt[c] for c in range(1, L + 1) if st[c] <= k - 1) % MOD for k in range(1, 6)]

rng = random.Random(4)
for _ in range(300):
    L = rng.randint(1, 11)
    s = '1' + ''.join(rng.choice('01') for _ in range(L - 1))
    k = rng.randint(1, 4)
    assert largest_below(s, k) == largest_brute(s, k), (s, k)
    assert all_k(s) == [Solution().countKReducibleNumbers(s, kk) for kk in range(1, 6)]
print('Bài 4, 5 khớp vét cạn. Ví dụ all_k("10110") =', all_k("10110"))

Bài 4, 5 khớp vét cạn. Ví dụ all_k("10110") = [5, 15, 21, 21, 21]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
steps[1] = 0; steps[c] = 1 + steps[popcount(c)]
cnt[c] = Σ trên i có s[i]=='1': C(L-i-1, c - ones_before_i)
ans = Σ_{c>=1, steps[c] <= k-1} cnt[c]  (mod 1e9+7)
```

**Checklist:**

- [ ] Điều kiện là `steps[c] ≤ k − 1` (đã dùng 1 thao tác để về `c`)?
- [ ] Chỉ đếm `x < n` — không cộng chính `n`?
- [ ] Bỏ `c = 0` (số 0)?
- [ ] Số bit tự do là `L − i − 1`?
- [ ] Modulo khi cộng tổ hợp?